# LCZ Map Acquisition — The Four Ways to Get an LCZ Map

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/general/01_map_acquisition.en.ipynb)

**Learning objective**: by the end of this notebook you will know how to fetch a Local Climate Zone (LCZ) classification map for any place on Earth using LCZ4py's four acquisition functions — the global open dataset, two validated regional variants (Europe, CONUS), and a user-submitted LCZ Generator job — and how to manage the local disk cache that makes repeated downloads instant.

## Why a standardized LCZ map?

The **Local Climate Zone (LCZ) scheme** (Stewart & Oke, 2012, *Bulletin of the American Meteorological Society*) was created to solve a chronic problem in urban climatology: studies described their study areas with informal, inconsistent labels — "urban", "suburban", "rural", "downtown" — that meant different things in different cities and made cross-study comparison of Urban Heat Island (UHI) intensities essentially impossible. Stewart & Oke proposed 17 standard zone types (classes 1–10 for built/urban surfaces — from compact high-rise to large low-rise industrial — and classes 11–17, labelled A–G, for natural land covers such as dense trees, low plants, bare rock, and water) defined by measurable surface properties: sky view factor, aspect ratio, building surface fraction, impervious surface fraction, height of roughness elements, and terrain roughness class. Two sites in the same LCZ class, anywhere in the world, are expected to have a similar thermal response to solar forcing, which is exactly what makes the scheme useful as a common currency for UHI studies, urban planning, and building energy modelling.

A **map** that assigns every pixel of a city (or a continent) to one of these 17 classes is what turns the scheme from theory into a usable spatial layer. Because LCZ maps are laborious to produce from scratch (they require satellite imagery, machine-learning classifiers, and expert validation), the community converged on a small number of open, versioned map products that researchers download rather than rebuild. LCZ4py — the Python port of the R package LCZ4r (Anjos et al., 2025, *Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0) — wraps the acquisition of all of them behind one consistent interface, so switching between a global overview and a locally validated regional product is a one-line change.

## The four acquisition paths in this notebook

1. **`lcz_get_map`** — the **global** LCZ map (Demuzere et al., 2022), a single worldwide mosaic at ~100 m resolution built from Sentinel-2 imagery and Google Earth Engine. Use this as the default choice for any city, anywhere: it guarantees the same methodology everywhere, which is what makes cross-city comparisons valid in the first place.
2. **`lcz_get_map_euro`** — the **European** LCZ map (Demuzere et al., 2019). A regional product with additional local validation and training data specific to European cities; where it overlaps with the global map, expect small class-boundary differences from the extra regional input data.
3. **`lcz_get_map_usa`** — the **CONUS** LCZ map (Demuzere et al., 2020), built on the NLCD (National Land Cover Database) and covering only the continental United States. It trades global coverage for US-specific validation and ancillary data.
4. **`lcz_get_map_generator`** — downloads a **specific, user-submitted job** from the [LCZ Generator](https://lcz-generator.rub.de/) web platform, identified by a job `id`. This is a fundamentally different acquisition path from the first three: instead of clipping a pre-existing global/regional mosaic to your area of interest, you are fetching one person's custom classification (their own training polygons, their own imagery date), which may cover a smaller or differently-shaped area and comes with its own accuracy assessment (visible on the factsheet page).

All three mosaic-based functions (`lcz_get_map`, `_euro`, `_usa`) share the same interface: pass a `city` name (geocoded automatically) or a custom `roi` GeoDataFrame, and get back the path to a clipped GeoTIFF. Under the hood they stream only the pixels that intersect your area of interest from a remote Cloud-Optimized GeoTIFF (COG) — no multi-gigabyte global file ever touches your disk — and cache the result locally so re-running the same city again is instant.

In [ ]:
pip install LCZ4py 

ERROR: Ignored the following versions that require a different python version: 0.1.0 Requires-Python >=3.10
ERROR: Could not find a version that satisfies the requirement LCZ4py (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for LCZ4py
Note: you may need to restart the kernel to use updated packages.


## 1. `lcz_get_map` — the global LCZ map

`lcz_get_map(city=None, roi=None, isave_map=False, cache=True, cache_dir="~/.lcz4r_cache", lang="en", verbose=True) -> str` is the main entry point of LCZ4py. Pass a `city` name (geocoded via Nominatim) or a custom `roi` GeoDataFrame; it returns the **path to a GeoTIFF** clipped to that area's boundary, with integer pixel values 1–17 (LCZ classes) and 0 for nodata.

Key parameters:
- `isave_map`: also copy the clipped raster into `LCZ4r_output/` (otherwise it lives only in the cache directory).
- `cache` / `cache_dir`: when `cache=True` (default), both the geocoded boundary (GeoJSON/GeoArrow) and the clipped raster (GeoTIFF) are saved under `cache_dir`, keyed by a hash of the city name and bounding box — running the same city again is near-instant. Set `cache=False` to force a fresh download into a temporary directory.
- `verbose`: print progress (geocoding, streaming, caching) to the log.

We use **Juiz de Fora** (the small capital of Liechtenstein) as the running example throughout this notebook: its tiny bounding box keeps every download to a few seconds, which is ideal for learning the API.

In [1]:
from LCZ4py import lcz_get_map
import rasterio
import numpy as np

map_path = lcz_get_map(city="Juiz de Fora", verbose=True)
print("Clipped LCZ map saved at:", map_path)

with rasterio.open(map_path) as src:
    arr = src.read(1)
    print("shape:", arr.shape, "| CRS:", src.crs, "| bounds:", src.bounds)
    classes, counts = np.unique(arr, return_counts=True)
    print("LCZ classes present (0 = nodata):", dict(zip(classes.tolist(), counts.tolist())))

12:54:15 - LCZ4py._internal._lcz_map_engine - INFO - Geocoding 'Juiz de Fora' via async HTTP...
12:54:16 - httpx - INFO - HTTP Request: GET https://nominatim.openstreetmap.org/search?q=Juiz+de+Fora&format=geojson&limit=1&polygon_geojson=1 "HTTP/1.1 200 OK"
12:54:16 - LCZ4py._internal._lcz_map_engine - INFO - Cached boundary to /Users/co2map/.lcz4r_cache/study_area_juiz_de_fora.arrow
12:54:16 - LCZ4py._internal._lcz_map_engine - INFO - Streaming COG and clipping to geometry...
12:54:41 - LCZ4py._internal._lcz_map_engine - INFO - Saved to clipped cache.


Clipped LCZ map saved at: /Users/co2map/.lcz4r_cache/clipped_c7031cb44f08aaf6.tif
shape: (534, 601) | CRS: EPSG:4326 | bounds: BoundingBox(left=-43.685970582016964, bottom=-21.99974130808667, right=-43.14608309626114, top=-21.520040946366848)
LCZ classes present (0 = nodata): {0: 165493, 2: 42, 3: 2852, 4: 4, 5: 40, 6: 6078, 8: 1007, 9: 17470, 10: 90, 11: 38366, 12: 78963, 13: 9, 14: 10066, 15: 50, 16: 11, 17: 393}


**Reading the output**: `map_path` points to a small GeoTIFF containing only the pixels that intersect Juiz de Fora's administrative boundary — everything outside it is nodata (0). The class histogram above tells you, at a glance, the urban morphology mix of the city: a high share of classes 1–10 (built types) versus 11–17 (natural/water types) signals a more built-up area, while a landscape dominated by classes like 11 (dense trees) or 15 (scattered trees) signals a mostly natural/rural surrounding. This is the raw input every other LCZ4py function in this series builds on.

## 2. `lcz_clear_cache` — purging the disk cache

`lcz_clear_cache(cache_dir=None) -> int` deletes every cached boundary (`study_area_*`) and clipped raster (`clipped_*`) file from the cache directory (default `~/.lcz4r_cache`, the same default used by `lcz_get_map`). It returns the **number of files deleted**. Use it when you want to force a fresh download for all cities (e.g. after a map source is updated), or simply to reclaim disk space.

In [3]:
from LCZ4py import lcz_clear_cache

n_deleted = lcz_clear_cache()
print(f"Deleted {n_deleted} cached file(s).")

Deleted 7 cached file(s).


**Reading the output**: the count reflects the boundary + raster cache entries removed (here, the ones created by the Juiz de Fora call above). The functions below will simply re-populate the cache on demand — clearing it never breaks anything, it only trades disk space for a slightly slower next download.

## 3. `lcz_get_map_euro` — the European LCZ map

`lcz_get_map_euro(city=None, roi=None, isave_map=False, isave_euro=False, cache=True, cache_dir=..., lang="en", verbose=True) -> str` has the exact same interface as `lcz_get_map`, plus an `isave_euro` flag kept only for API parity with the R package (the Python COG-streaming implementation makes saving the full European mosaic unnecessary, so it is a no-op). It sources from the **Demuzere et al. (2019)** European LCZ map instead of the global mosaic. Use this variant for European cities when you want the extra regional validation baked into that product, at the cost of only covering Europe.

In [ ]:
from LCZ4py import lcz_get_map_euro

map_path_euro = lcz_get_map_euro(city="Barcelona", verbose=True)
print("Clipped European LCZ map saved at:", map_path_euro)

with rasterio.open(map_path_euro) as src:
    arr_euro = src.read(1)
    classes, counts = np.unique(arr_euro, return_counts=True)
    print("shape:", arr_euro.shape, "| CRS:", src.crs)
    print("LCZ classes present (0 = nodata):", dict(zip(classes.tolist(), counts.tolist())))

06:26:35 - LCZ4py._internal._lcz_map_engine - INFO - Geocoding 'Juiz de Fora' via async HTTP...


06:26:35 - httpx - INFO - HTTP Request: GET https://nominatim.openstreetmap.org/search?q=Juiz de Fora&format=geojson&limit=1&polygon_geojson=1 "HTTP/1.1 200 OK"


06:26:35 - LCZ4py._internal._lcz_map_engine - INFO - Cached boundary to /Users/co2map/.lcz4r_cache/study_area_vaduz.arrow


06:26:35 - LCZ4py._internal._lcz_map_engine - INFO - Streaming COG and clipping to geometry...


06:26:40 - rasterio._env - WARNING - CPLE_IllegalArg in tmpphwk63xe.tif: BLOCKXSIZE can only be used with TILED=YES


06:26:40 - LCZ4py._internal._lcz_map_engine - INFO - Saved to clipped cache.


Clipped European LCZ map saved at: /Users/co2map/.lcz4r_cache/clipped_8755df8f6d014dec.tif
shape: (80, 87) | CRS: EPSG:4326
LCZ classes present (0 = nodata): {0: 5830, 5: 2, 6: 181, 8: 3, 9: 36, 11: 586, 12: 151, 14: 88, 15: 3, 17: 80}


**Reading the output**: compare this class histogram with the global-map one above. Small differences in class boundaries or class assignment between the two are expected and reflect the extra European-specific training/validation data used to build this regional product — neither is "wrong", they are simply different snapshots of the same underlying reality with different local calibration.

## 4. `lcz_get_map_usa` — the CONUS LCZ map

`lcz_get_map_usa(city=None, roi=None, isave_map=False, isave_usa=False, cache=True, cache_dir=..., lang="en", verbose=True) -> str` mirrors the same interface again, this time sourcing from the **Demuzere et al. (2020)** CONUS (CONtinental United States) LCZ map, built on NLCD land-cover data. Because its raster mosaic only covers the continental US, **this function requires a US city or ROI** — unlike `lcz_get_map`/`lcz_get_map_euro`, Juiz de Fora (Liechtenstein) would fall entirely outside its extent and raise an error. We switch the running example to a small US town, **Truckee, California**, to keep the download just as fast.

In [ ]:
from LCZ4py import lcz_get_map_usa

map_path_usa = lcz_get_map_usa(city="Chicago", verbose=True)
print("Clipped CONUS LCZ map saved at:", map_path_usa)

with rasterio.open(map_path_usa) as src:
    arr_usa = src.read(1)
    classes, counts = np.unique(arr_usa, return_counts=True)
    print("shape:", arr_usa.shape, "| CRS:", src.crs)
    print("LCZ classes present (0 = nodata):", dict(zip(classes.tolist(), counts.tolist())))

06:26:40 - LCZ4py._internal._lcz_map_engine - INFO - Geocoding 'Truckee, California' via async HTTP...


06:26:40 - httpx - INFO - HTTP Request: GET https://nominatim.openstreetmap.org/search?q=Truckee%2C+California&format=geojson&limit=1&polygon_geojson=1 "HTTP/1.1 200 OK"


06:26:40 - LCZ4py._internal._lcz_map_engine - INFO - Cached boundary to /Users/co2map/.lcz4r_cache/study_area_truckee,_california.arrow


06:26:40 - LCZ4py._internal._lcz_map_engine - INFO - Streaming COG and clipping to geometry...


06:26:44 - rasterio._env - WARNING - CPLE_IllegalArg in tmpb3kzgv10.tif: BLOCKXSIZE can only be used with TILED=YES


06:26:44 - LCZ4py._internal._lcz_map_engine - INFO - Saved to clipped cache.


Clipped CONUS LCZ map saved at: /Users/co2map/.lcz4r_cache/clipped_05b9aad9021c0c57.tif
shape: (66, 208) | CRS: EPSG:4326
LCZ classes present (0 = nodata): {0: 6014, 6: 2259, 11: 600, 12: 4242, 13: 256, 14: 20, 16: 12, 17: 325}


**Reading the output**: Truckee is a small mountain town, so expect the histogram to skew heavily toward natural classes (e.g. 12/dense trees, 13/scattered trees, 15/water or bare rock at elevation) with only a thin slice of low-rise/open built classes (5, 6, 8) near the town center — a useful sanity check that the CONUS product is behaving as expected for a small, low-density US settlement.

## 5. `lcz_get_map_generator` — a specific LCZ Generator job

`lcz_get_map_generator(id="3110e623fbe4e73b1cde55f0e9832c4f5640ac21", band="lczFilter", isave_map=False, save_extension="tif") -> str` is a different kind of acquisition: instead of clipping a global/regional mosaic to your area, it downloads **one specific classification job** submitted by a user to the [LCZ Generator](https://lcz-generator.rub.de/) web platform, identified by its `id` (visible in the job's public factsheet URL). `band` selects which output layer to keep: `"lcz"` (the raw classifier output) or `"lczFilter"` (default — the post-processed, spatially filtered/smoothed map, generally preferred). The default `id` used below is a public example job, so it works without you having ever submitted anything yourself. This path matters when you need a classification with a specific accuracy assessment, training date, or custom training polygons that neither the global nor a regional mosaic can offer.

In [6]:
from LCZ4py import lcz_get_map_generator

map_path_gen = lcz_get_map_generator()  # default id: a public example job
print("LCZ Generator map saved at:", map_path_gen)

with rasterio.open(map_path_gen) as src:
    arr_gen = src.read(1)
    classes, counts = np.unique(arr_gen, return_counts=True)
    print("shape:", arr_gen.shape, "| CRS:", src.crs)
    print("LCZ classes present (0 = nodata):", dict(zip(classes.tolist(), counts.tolist())))

06:26:45 - LCZ4py.general.lcz_get_map_generator - INFO - Streaming map from LCZ Generator...


06:26:46 - httpx - INFO - HTTP Request: GET https://lcz-generator.rub.de/factsheets/3110e623fbe4e73b1cde55f0e9832c4f5640ac21/3110e623fbe4e73b1cde55f0e9832c4f5640ac21.tif "HTTP/1.1 200 OK"


LCZ Generator map saved at: /var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmpkwaj1511.tif
shape: (727, 1435) | CRS: EPSG:4326
LCZ classes present (0 = nodata): {0: 2870, 1: 2275, 2: 767, 3: 61473, 4: 155, 5: 63, 6: 63777, 7: 1687, 8: 5710, 9: 44621, 10: 207, 11: 244735, 12: 43415, 13: 25230, 14: 214151, 15: 518, 16: 3191, 17: 328400}


**Reading the output**: this raster typically covers a much larger extent than the tiny Juiz de Fora/Truckee clips above, since LCZ Generator jobs are usually submitted for whole metropolitan regions. Note that unlike the three mosaic functions, there is no `city`/`roi` parameter here — the spatial extent is whatever the original submitter defined for their job, not something you control from LCZ4py.

## Recap & next steps

You now have four ways to put an LCZ map on disk: `lcz_get_map` (global default), `lcz_get_map_euro` / `lcz_get_map_usa` (regional variants with extra local validation, at the cost of restricted coverage), and `lcz_get_map_generator` (a specific user-submitted job). All three mosaic functions share a disk cache you can inspect and purge with `lcz_clear_cache`.

Every GeoTIFF path returned here (`map_path`, `map_path_euro`, `map_path_usa`, `map_path_gen`) is exactly the kind of input the rest of the LCZ4py general-purpose toolkit expects. **Next: `02_visualization_area_stats`** — plotting these maps interactively and computing the area occupied by each LCZ class.